**Descarga de librerías**

In [ ]:
!pip install -qqq arxiv pandas torch sentence-transformers faiss-cpu
!pip install -qqq langchain langchain-community langchain-core langchain_text_splitters
!pip install -qqq umap-learn matplotlib

ERROR: Operation cancelled by user



## CREACIÓN BASE DE DATOS



**Obtención abstracts y limpieza mínima**

El dataset.csv se puede encontrar en la carpeta de archivos.

In [ ]:
import arxiv
import pandas as pd
import re

# Limpieza mínima
def clean_text(text):
  text = text.replace("\n", " ") # quitar saltos de línea
  text = re.sub(r'\s+', ' ', text) # quitar espacios duplicados
  return text.strip()

# Búsqueda en arXiv
client = arxiv.Client()

search = arxiv.Search(
    query='Spin Hall effect, Rashba effect',
    max_results=200,
    sort_by=arxiv.SortCriterion.Relevance)

data = []
for result in client.results(search):
    abstract = clean_text(result.summary)

    data.append({
        "id": result.entry_id,
        "title": result.title,
        "abstract": abstract,
        "year": result.published.year,
    })

# Crear DataFrame
df = pd.DataFrame(data)
df = df.dropna(subset=["abstract"])
df = df.drop_duplicates(subset=["title"])

# Guardar dataset
df.to_csv("dataset.csv", index=False)
print(f"Dataset guardado con {len(df)} abstracts")
print(df.head())


Dataset guardado con 200 abstracts
                                  id  \
0  http://arxiv.org/abs/2604.25786v1   
1  http://arxiv.org/abs/2604.24358v1   
2  http://arxiv.org/abs/2604.23093v1   
3  http://arxiv.org/abs/2604.22573v1   
4  http://arxiv.org/abs/2604.16803v1   

                                               title  \
0  Homogeneous Stellar Parameters from Heterogene...   
1  Recovering extra-tidal open cluster members vi...   
2  Analysis of a Septuple Open Cluster System and...   
3  A homogeneous three-dimensional view of Molecu...   
4  W UMa-Type Contact Binaries in the Tidal Tails...   

                                            abstract  year  
0  Large-scale spectroscopic surveys have collect...  2026  
1  The identification of open cluster (OC) member...  2026  
2  A rare multiple open cluster system has been a...  2026  
3  Understanding the large-scale dynamics of mole...  2026  
4  Star clusters, as dynamically rich environment...  2026  


**Chunking y Embedding de los abstracts**

En la carpeta de archivos aparecerá dos ficheros de los chunks (uno .csv y uno .json) además del índice faiss.

In [ ]:
import pandas as pd
import faiss
import numpy as np
import json

from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Cargar dataset
df = pd.read_csv("dataset.csv")
df = df.dropna(subset=["abstract"]) # Así evitamos NaN

# CHUNKING
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100 )

all_chunks = []
for i, (_, row) in enumerate(df.iterrows()):
    text = row["title"] + ". " + row["abstract"]
    chunks = splitter.split_text(text)

    for j, chunk in enumerate(chunks):
        all_chunks.append({
            "chunk_id": len(all_chunks),
            "text": chunk,
            "title": row["title"],
            "year": row["year"],
            "full_text": text,
            "id": row["id"]
        })

chunk_df = pd.DataFrame(all_chunks)
chunk_df = chunk_df.reset_index(drop=True)

print(f"Total chunks: {len(chunk_df)}")

# EMBEDDINGS
model = SentenceTransformer('BAAI/bge-base-en-v1.5')

texts = chunk_df["text"].tolist()

embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

embeddings = np.array(embeddings).astype("float32")

# FAISS
dimension = embeddings.shape[1]

index = faiss.IndexHNSWFlat(dimension, 32)
index.metric_type = faiss.METRIC_INNER_PRODUCT

index.add(embeddings)

print(f"Índice FAISS creado con {index.ntotal} vectores")

# MAPPING
chunk_df.to_csv("chunks.csv", index=False) # CSV Relación índice-texto

mapping = chunk_df.to_dict(orient="records") # JSON (mapping real)

with open("chunk_mapping.json", "w") as f:
    json.dump(mapping, f, indent=2)

# Guardar índice FAISS
faiss.write_index(index, "faiss_index.bin")



Total chunks: 520


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Índice FAISS creado con 520 vectores


## PIPELINE

**Pregunta y respuesta** con FLAN T5

In [ ]:
import faiss
import numpy as np

from sentence_transformers import CrossEncoder
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda

# LLM
model_name = "google/flan-t5-large"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model_llm = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# PROMPT
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are a scientific assistant.

Use the context to answer the question.
You must rephrase and summarize the information.

If the answer is not present, say "Not enough information".

Context:
{context}

Question:
{question}

Answer in your own words:
"""
)

# GENERATION FUNCTION
def llm_generate(inputs):
    prompt_text = prompt.format(**inputs)

    tokens = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = model_llm.generate(
        **tokens,
        max_new_tokens=150
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

rag_chain = RunnableLambda(llm_generate)

# RERANKER
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# RAG PIPELINE
def rag(question):

    # -------- EMBEDDING --------
    question_emb = model.encode([question],
        normalize_embeddings=True
    ).astype("float32")

    # -------- RETRIEVAL (FAISS) ---------
    D, I = index.search(question_emb, 5)

    retrieved = []
    print("\n=== FAISS RESULTS ===\n")

    for score, i in zip(D[0], I[0]):
        item = chunk_df.iloc[i]

        print(f"FAISS score: {score:.4f} | {item['title']}")

        retrieved.append({
            "text": item["text"],
            "title": item["title"],
            "year": item["year"],
            "faiss_score": score,
            "id": item["id"] })

    # -------- RERANKING --------
    pairs = [[question, c["text"]] for c in retrieved]
    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(scores, retrieved),
        key=lambda x: x[0],
        reverse=True)

    top_chunks = [c for _, c in ranked[:3]]

    # -------- CONTEXT BUILDING --------
    context = "\n\n".join([
        f"[Paper: {c['title']} ({c['year']})]\n{c['text'][:300]}"
        for c in top_chunks ])

    print("\n=== CONTEXTO UTILIZADO ===\n")
    print(context)

    # -------- GENERATION --------
    generated_response = rag_chain.invoke({
        "context": context,
        "question": question })
    return generated_response, context, retrieved

# QUERY
question = "Which stellar parameters does Gaia DR3 measure?"
response, context, retrieved = rag(question)

print("\n=== RESPUESTA FINAL ===\n")
print(response)


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]


=== FAISS RESULTS ===

FAISS score: -0.3754 | The Gaia-ESO Survey DR5.1 and Gaia DR3 GSP-Spec: a comparative analysis
FAISS score: -0.4176 | Deep Photometric and Astrometric Investigation of the Non-relaxed Star Cluster Stock 3 using Gaia DR3
FAISS score: -0.4544 | Selected open cluster sample for validating atmospheric parameters: Application to Gaia and other surveys
FAISS score: -0.4678 | Deeply Comprehensive Astrometric, Photometric, and Kinematic Studies of the Three OCSN Open Clusters with Gaia DR3
FAISS score: -0.4790 | The Gaia-ESO Survey DR5.1 and Gaia DR3 GSP-Spec: a comparative analysis

=== CONTEXTO UTILIZADO ===

[Paper: The Gaia-ESO Survey DR5.1 and Gaia DR3 GSP-Spec: a comparative analysis (2024)]
The Gaia-ESO Survey DR5.1 and Gaia DR3 GSP-Spec: a comparative analysis. (abridged) The third data release of Gaia, has provided stellar parameters, metallicity [M/H], [α/Fe], individual abundances, broadening parameter from its RVS spectra for about 5.6 million objects thanks

## ANÁLISIS DE RESULTADOS

**Gráfica UMAP con el query y contexto recuperado**

In [ ]:
# @title
# The original attempt to install umap-learn==0.6.0 failed because it's not compatible with Python 3.12.
# Instead, we will keep the already installed umap-learn (likely 0.5.12) and upgrade its dependency pynndescent
# to a version compatible with the current Numba (0.60.0).
!pip install pynndescent==0.7.0
import umap
import matplotlib.pyplot as plt


# QUERY

#question = "Does Gaia DR3 include radial velocities for all stars?"
question = "Which stellar parameters does Gaia DR3 measure?"
#question = "How has the formation of contact binaries been investigated?"

question_emb = model.encode(
    [question],
    normalize_embeddings=True
).astype("float32")

# =========================
# FAISS RETRIEVAL
# =========================
k = 10
D, I = index.search(question_emb, k)
retrieved_indices = I[0]

# =========================
# UMAP
# =========================
reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",
    random_state=42
)

emb_2d = reducer.fit_transform(embeddings)
question_2d = reducer.transform(question_emb)


# PLOT
plt.figure(figsize=(10,7))

# corpus
plt.scatter(emb_2d[:,0], emb_2d[:,1], alpha=0.2)

# retrieved
plt.scatter(
    emb_2d[retrieved_indices,0],
    emb_2d[retrieved_indices,1],
    marker="x",
    s=120,
    label="Retrieved"
)

# query
plt.scatter(
    question_2d[:,0],
    question_2d[:,1],
    marker="*",
    s=300,
    label="Question"
)

plt.legend()
plt.title("UMAP: Question vs Retrieved Chunks")
plt.show()

ERROR: Could not find a version that satisfies the requirement pynndescent==0.7.0 (from versions: 0.1.0, 0.1.1, 0.1.2, 0.2.0, 0.2.1, 0.3.0, 0.3.2, 0.3.3, 0.4.0, 0.4.1, 0.4.2, 0.4.3, 0.4.4, 0.4.5, 0.4.6, 0.4.7, 0.4.8, 0.5.0rc1, 0.5.0, 0.5.1, 0.5.2, 0.5.3, 0.5.4, 0.5.5, 0.5.6, 0.5.7, 0.5.8, 0.5.9, 0.5.10, 0.5.11, 0.5.12, 0.5.13, 0.6.0)
ERROR: No matching distribution found for pynndescent==0.7.0


TypingError: Failed in nopython mode pipeline (step: nopython frontend)
Unknown attribute 'empty_list' of type typeref[<class 'numba.core.types.containers.ListType'>]

File "../usr/local/lib/python3.12/dist-packages/pynndescent/sparse.py", line 230:
def sparse_mul(ind1, data1, ind2, data2):
    result_ind = numba.typed.List.empty_list(numba.types.int32)
    ^

During: typing of get attribute at /usr/local/lib/python3.12/dist-packages/pynndescent/sparse.py (230)

File "../usr/local/lib/python3.12/dist-packages/pynndescent/sparse.py", line 230:
def sparse_mul(ind1, data1, ind2, data2):
    result_ind = numba.typed.List.empty_list(numba.types.int32)
    ^


## Evaluación personalizada con preguntas y análisis de FAISS

In [ ]:
# Definir preguntas de prueba aquí

custom_questions = [
    #"How does the molecular cloud density vary with Galactic longitude?"
    #  "How has the release of Gaia data impact the number of known Galactic open clusters?",
   #"What are open clusters widely considered as potential enviroments for?"
      # "How has the formation of contact binaries been investigated?",
    #"Which stellar parameters does Gaia DR3 measure?",
    #"Does Gaia DR3 include radial velocities for all stars?",
    #"What is the typical metallicity of open clusters studied with Gaia?"
    "as YSOs, and to reconstruct the three-dimensional (3D) motions of the main MC complexes within 2.5 kpc of the Sun using YSOs and young OCs as tracers. Using Gaia DR3 astrometry together with complementary spectroscopic surveys for radial velocities, we compiled a unified sample of 24,732 stellar tracers. We applied robust clustering in proper motion space to identify co-moving YSOs and derived cloud-averaged motions via Monte Carlo sampling. These were compared with the kinematics of OCs younger than 30 Myr. Finally, we performed orbital integrations in a realistic Galactic potential to tra"
    #"n high-resolution APOGEE spectra the model achieves precisions of $18~$K in $\textrm{T}_{\rm eff}$, $0.04~$dex in $\textrm{log}\,\textit{g}$, $0.015~$dex in [Fe/H], and ${<}\,0.03~$dex across all abundances; on lower-resolution DESI spectra, typical precisions are $51~$K, $0.09~$dex, $0.04~$dex, and ${\sim}\,0.06~$dex, respectively. Cross-survey comparisons demonstrate that labels for the same stars observed by different surveys are"
]

for i, q in enumerate(custom_questions):
    print(f"\n--- Evaluando Pregunta {i+1}: {q} ---\n")
    response, context, retrieved_faiss = rag(q)

    print("\n=== RESULTADOS DE FAISS (antes del re-ranking) ===\n")
    for item in retrieved_faiss:
        print(f"  Título: {item['title']} ({item['year']})")
        print(f"  Puntuación FAISS: {item['faiss_score']:.4f}")
        print(f"  Extracto: {item['text'][:150]}...")
        print("  ---")

    print("\n=== CONTEXTO FINAL (después del re-ranking) ===\n")
    print(context)

    print("\n=== RESPUESTA DEL MODELO ===\n")
    print(response)
    print("\n======================================================\n")


--- Evaluando Pregunta 1: as YSOs, and to reconstruct the three-dimensional (3D) motions of the main MC complexes within 2.5 kpc of the Sun using YSOs and young OCs as tracers. Using Gaia DR3 astrometry together with complementary spectroscopic surveys for radial velocities, we compiled a unified sample of 24,732 stellar tracers. We applied robust clustering in proper motion space to identify co-moving YSOs and derived cloud-averaged motions via Monte Carlo sampling. These were compared with the kinematics of OCs younger than 30 Myr. Finally, we performed orbital integrations in a realistic Galactic potential to tra ---


=== FAISS RESULTS ===

FAISS score: -0.0206 | A homogeneous three-dimensional view of Molecular Cloud kinematics out to 2.5 kpc. Using Young Stellar Objects and Open Clusters as complementary tracers
FAISS score: -0.2354 | A homogeneous three-dimensional view of Molecular Cloud kinematics out to 2.5 kpc. Using Young Stellar Objects and Open Clusters as complementary 